In [1]:
import pandas as pd
import os
import pickle

In [2]:
from nmf_utils import compute_histogram_df

In [3]:
def main(path_save, file_stem):
    """
    Compute the average time-to-peak histogram for a recording.

    This function:
    1. Loads peak analysis results from a saved pickle bundle.
    2. Extracts time-to-peak values for all ROIs.
    3. Computes a normalized histogram per ROI using fixed binning.
    4. Averages these histograms across ROIs to obtain a population-level distribution.

    Parameters
    ----------
    path_save : str
        Directory where the peak results file is stored.
    file_stem : str
        Base filename used to locate the peak results file.

    Returns
    -------
    hist_df : pd.Series
        Mean normalized histogram of time-to-peak values:
        - Index: bin centers (in seconds)
        - Values: average probability per bin across ROIs

    Notes
    -----
    - Histogram bins are defined from -0.05 to 5.05 seconds with 0.1 s width.
    - Each ROI contributes equally after normalization.
    - Output represents the population-level distribution of event kinetics.
    """
    #Load peak timing data
    file_data = os.path.join(path_save, f"{file_stem}_peaks.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    time_to_peak = bundle["time_to_peak"]

    # Compute histogram 
    hist_df = compute_histogram_df(
        time_to_peak, 
        bin_start=-0.05, 
        bin_end=5.05, 
        step=0.1
        )

    return hist_df
   

 

In [4]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [5]:
path_dist = "Data"
# Load genotype information (indexed by pup ID)
genotype=pd.read_excel(os.path.join(path_dist, "genotypes_exp1_12.xlsx"), index_col=0)

# Initialize DataFrame to collect histogram results
# Columns = pups (samples), index = bin centers (time-to-peak bins
all_hist_df = pd.DataFrame()

#Loop over experiments and recordings
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        # Create output directory for this recordin
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        # Construct sample identifier (pup ID)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        # Compute mean histogram for this recording
        hist_df= main(path_save,file_stem)
        # Store histogram as a column (one column per sample
        all_hist_df[pup] = hist_df
        
        #Compute grouped statistics by genotype, transpose -> rows = samples, columns = bins 
        mean_per_genotype = all_hist_df.T.groupby(genotype["genotype"]).mean().T
        sd_per_genotype = all_hist_df.T.groupby(genotype["genotype"]).std().T
        n_per_genotype = all_hist_df.T.groupby(genotype["genotype"]).count().T

        print(f"Processing {file_stem} from {folder_exp}")

#Save aggregated results
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
mean_per_genotype.to_excel(os.path.join(path_dist, "NMF_analysis_results", "nmf_compounds_t2p_hist_mean.xlsx"))
sd_per_genotype.to_excel(os.path.join(path_dist, "NMF_analysis_results", "nmf_compounds_t2p_hist_sd.xlsx"))
n_per_genotype.to_excel(os.path.join(path_dist, "NMF_analysis_results", "nmf_compounds_t2p_hist_n.xlsx"))

Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
